In [1]:
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass

import math
torch.manual_seed(1337)



In [2]:
@dataclass
class GPTConfig:
    block_size: int = 512   # 这里其实应该是文本的最大长度（ max_seq_len）
    batch_size: int = 12
    n_layer: int = 6
    n_head: int = 12
    n_embd: int = 768    # n_embd 也叫 hidden_dim, hiden_size, 这里我同时设置了和 embed_dim 一样
    hidden_dim: int = n_embd
    
    head_size = n_embd // n_head
    dropout: float = 0.1
    # # tiktoken 使用的是 GPT-2 的词表，大约有 50257 个token
    vocab_size: int = 50257



In [3]:
class SingleHeadAttention(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.key = nn.Linear(config.hidden_dim, config.head_size)
        self.value = nn.Linear(config.hidden_dim, config.head_size)
        self.query = nn.Linear(config.hidden_dim, config.head_size)

        # 尝试学习新的写法，attention_mask 通过 register_buffer 注册
        # 因为不用计算 梯度，所以节约内存和显存，速度也更快
        self.register_buffer(
            'attention_mask',
            # tril是下三角矩阵
            torch.tril(
                torch.ones(config.block_size, config.block_size)
            )
        )
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        batch_size, seq_len, hidden_dim = x.size()
        k = self.key(x)
        v = self.value(x)
        q = self.query(x)
        weight = q @ k.transpose(-2, -1)
        weight = weight.masked_fill(
            ## 如果文本长度没到 block_size，那么这里会结合pad_mask进行使用
            self.attention_mask[:seq_len, :seq_len] == 0,
            float('-inf')
        )
        ## 这里计算attention的权重
        ## 需要对每个query,计算与所有key位置权重
        ## 因此设置dim = -1
        weight = F.softmax(weight, dim = -1) / math.sqrt(self.head_size)
        
        ## dropout要放到attention_weight，
        ## 而不是放在乘上v之后的value上
        weight = self.dropout(weight)
        out = weight @ v
        return out


class MultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.heads = nn.ModuleList(
            [
                SingleHeadAttention(config)
                for _ in range(config.n_head)
            ]
        )
        ## 多头attention一共有4个投影矩阵
        ## 分别是 输入到query, key, value 的投影矩阵
        ## 以及 多头attention 的输出到最终输出的投影矩阵
        self.proj = nn.Linear(config.n_embd, config.n_embd)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        output = torch.cat(
            [h(x) for h in self.heads],
            dim = -1
        )
        output = self.proj(output)
        output = self.dropout(output)
        return output
    
class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.net = nn.Sequential(
             nn.Linear(config.n_embd, 4 * config.n_embd),
             nn.GELU(),
             nn.Linear(4 * config.n_embd, config.n_embd),
             nn.Dropout(config.dropout)
        )
    
    def forward(self, x):
        return self.net(x)


# 接下来就是一个完整的 Block

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        head_size = config.n_embd // config.n_head
        self.attn = MultiHeadAttention(config)
        self.ffm = FeedForward(config)
        self.ln1 = nn.LayerNorm(config.n_embd)
        self.ln2 = nn.LayerNorm(config.n_embd)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x
    


# 完整的  GPT model
class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        ## (embedding, positiom, norm ,mlp ,block)
        ## 更新一点的大模型
        ## 将position embedding 从 0，1，xxx embedding 到 POPE
        ## 从layer Nomr -> RMSNorm
        ## 从mlp -> swiGLU
        ## MHA->GPA/ MLA
        self.token_embedding_table = nn.Embedding(config.vocab_size, config.n_embd)
        self.position_embedding_table = nn.Embedding(config.block_size, config.n_embd)
        self.blocks = nn.Sequential(
            *[Block(config) for _ in range(config.n_layer)]
        )
        self.ln_final = nn.LayerNorm(config.n_embd)
        ## 最后这个lm_head 是 线性层，
        ## 马上要做softmax，所以bias=False
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        
        ## 现在的SLM模型中，会用tie weight 来减少参数
        ## 通过减少embedding中的参数，来增加主干网络的参数
        self.token_embedding_table.weight = self.lm_head.weight

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        # idx 是输入的 token ids
        # targets 是 目标token ids
        # shape要一样
        batch, seq_len = idx.size()
        token_emb = self.token_embedding_table(idx) # (batch, seq_len, n_embd)

        pos_emb = self.position_embedding_table(
            torch.arrange(seq_len, device = idx.device)
        ) # (seq_len, n_embd)
        # 有一个经典题目：为什么 embedding 和 position 可以相加？
        x = token_emb + pos_emb # shape is (batch, seq_len, n_embd)
        x = self.blocks(x)  
        x = self.ln_final(x)
        logits = self.lm_head(x)  # shape is (batch, seq_len, vocab_size)

        if targets is not None:
            loss = None
        else :
            batch, seq_len, vocab_size = logits.size()
            logits = logits.view(batch * seq_len, vocab_size)
            targets = targets.view(batch * seq_len)
            loss = F.cross_entropy(logits, targets)
        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        pass



In [4]:

# 写一个 dataset，为了 Dataloader 准备
class MyDataset(Dataset):
    
    def __init__(self, path, block_size=512):

        import tiktoken
        self.enc = tiktoken.get_encoding("gpt2")
        self.block_size = block_size
        
        ## 特殊符号用于分割训练文本 [50256]
        self.eos_token = self.enc.encode(
            "<|endoftext|>",
            allowed_special={"<|endoftext|>"}
        )[0] ##返回的是一个list, 取第一个元素

        import json
        
        self.max_lines = 1000
        raw_data = []

        ## 在训练前，提前把数据处理好
        ## 转化为一定长度的文本，并tokenizer化
        ## 相较于transformers库的trainer
        ## 中边训练边处理，训练会更快
        with open(path, 'r') as f:
            for i,line in f:
                if i >= self.max_lines:
                    break
                try:
                    text = json.loads(line.strip())['text']
                    raw_data.append(text)
                except json.JSONDecodeError as je:
                    print(f"JSON解析错误，跳过行 {i}: {je}")
                    continue
                except Exception as e:
                    print(f"未知错误，跳过行 {i}: {e}")
                    continue
        
        full_encoded = []

        ## 预训练中很重要的一个技术是
        ## 把所有文本拼接起来，用特殊符号进行分割
        for text in raw_data:
            encoded_text = self.enc.encode(text) # list
            full_encoded.extend(encoded_text + [self.eos_token])
        
        ## 这里设置的block_size 是 512
        ## 把长文本切割为 512的文本块
        for i in range(0, len(full_encoded), self.block_size):
            # 多取一个 Token 作为目标
            ## !!这里多取一个 Token , 方便对target进行shift移位处理
            ## 1，2，3 -> 2，3，4
            chunk = full_encoded[i:i + self.block_size + 1] #每一行实际是513
            # 如果长度不够，用 eos_token 填充
            if len(chunk) < self.block_size + 1:
                chunk = chunk + [self.eos_token] * (self.block_size + 1 - len(chunk))
            self.encoded_data.append(chunk)


    def __len__(self):
        return len(self.encoded_data)
    
    def __getitem__(self, idx):
        chunk = self.encoded_data[idx]
        x = torch.tensor(chunk[:-1], dtype=torch.long) #torch.longw
        y = torch.tensor(chunk[1:], dtype=torch.long)  #是64位整数
        return x, y
    
    def encode(self, text):
        """将文本编码为token IDs"""
        return self.enc.encode(text)

    def decode(self, ids):
        """将token IDs解码为文本"""
        return self.enc.decode(ids)



In [7]:
# 1. 模型初始化
model = GPT(GPTConfig())
#使用自定义的GPT类创建模型实例
# GPTConfig() 是模型的配置类，包含各种超参数

# 2. 设备选择
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# 3. 模型迁移到设备
model = model.to(device)
# .to(device) 将模型的所有参数和缓冲区移动到指定设备
# 这是PyTorch中非常重要的操作，确保计算在正确的设备上进行

# 4. 计算模型参数数量
total_params = sum(p.numel() for p in model.parameters())
# model.parameters() 返回模型的所有可训练参数
# numel() 返回张量中元素的总数
# sum() 计算所有参数的总数
print(f"模型总参数量是：{total_params / 1e6}M")

# 5. 优化器设置
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

# 6. 学习率调度器
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=1000)
# CosineAnnealingLR 实现余弦退火学习率调度
# T_max=1000 设置余弦周期的长度



模型总参数量是：81.51936M


In [ ]:
# 训练循环
def train(model, optimizer, scheduler, train_loader, val_loade, device):
    model.train() # 设置模型为训练模式
    total_loss= 0.0
    for batch_idx, (x, y) in enumerate(train_loader):
        # 将数据移到设备上
        x, y = x.to(device), y.to(device)

        # 前向传播
        logits, loss = model(x, traget=y)

        # 反向传播
        optimizer.zero_grad()
        loss.backward()
        # 调整学习率
        optimizer.step()

        # 更新学习率
        scheduler.step()

        total_loss += loss.item()

        if batch_idx % 10 == 0:
            print(f'Epoch {epoch}, Batch {batch_idx}, Loss: {loss.item()}')

def eval(model, val_loader, device):
    # 验证
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            logits, loss = model(x, target=y)
            val_loss += loss.item() # 从张量中取纯数值
    return val_loss

for epoch in range(2):
    train_loss = train(model, optimizer, scheduler, train_loader, val_loader, device)
    val_loss = eval(model, val_loader, device)
    ##模型在训练数据上的平均损失值和验证损失
    print(f'Epoch: {epoch}, Train Loss: {train_loss/len(train_loader):.4f}, Val Loss: {val_loss/len(val_loader):.4f}')

    # 保存模型
    avg_val_loss = val_loss / len(val_loader)
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'val_loss': avg_val_loss,
    }
    # 保存每个epoch的模型
    torch.save(checkpoint, f'checkpoints/model_epoch_{epoch}.pt')